In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# --- Configuration ---
model_name = "/data/projects/mayapeng/hosptials_model/DeepSeek-R1-Distill-Qwen-14B/pretrain/"
dataset_path = "/data/projects/mayapeng/hosptials_model/DeepSeek-R1-Distill-Qwen-14B/train.json"
output_dir = "/data/projects/mayapeng/hosptials_model/DeepSeek-R1-Distill-Qwen-14B/train_lora_res2/"

# --- 1. Load model and tokenizer (Full precision bfloat16, no quantization) ---
print("🔄 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
# Enable gradient checkpointing to save GPU memory (allows for larger batch size or longer sequences)
model.gradient_checkpointing_enable()
print("✅ Model loaded successfully.")

# --- 2. Data Preprocessing (Increase max_length) ---
def format_instruction(example):
    if "input" in example and example["input"]:
        text = f"### 指令:\n{example['instruction']}\n\n### 输入:\n{example['input']}\n\n### 回答:\n{example['output']}"
    else:
        text = f"### 指令:\n{example['instruction']}\n\n### 回答:\n{example['output']}"
    return {"text": text}

def preprocess_function(examples):
    texts = [format_instruction({"instruction": inst, "input": inp, "output": out})["text"]
             for inst, inp, out in zip(examples["instruction"], 
                                        examples.get("input", [""]*len(examples["instruction"])), 
                                        examples["output"])]
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=5120,          # Adjust based on your data length (original data approx. 2700 tokens)
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# --- 3. Load dataset ---
print("🔄 Loading dataset...")
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"✅ Dataset loaded, total {len(dataset)} records.")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("🔄 Tokenizing data...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)
print("✅ Data tokenization completed.")

# --- 4. Enhanced LoRA Configuration (Adjusted to match the criteria) ---
print("🔄 Configuring Enhanced LoRA...")
# Linear layer names for Qwen models (can be checked via model.named_modules())
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",      # attention
    "gate_proj", "up_proj", "down_proj",         # MLP
]

lora_config = LoraConfig(
    r=32,                        # Set rank to 32 as specified in the criteria
    lora_alpha=64,               # Scaling factor set to 64 as specified in the criteria
    target_modules=target_modules,
    modules_to_save=["embed_tokens", "lm_head"],  # Fully fine-tune embedding layer and language modeling head
    lora_dropout=0.1,            # Dropout rate 0.1
    bias="all",                  # Train all bias terms
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Trainable parameter ratio will increase to approx. 1~3%
print("✅ LoRA configuration completed.")

# --- 5. Training Arguments (A100 80GB can accommodate a larger batch size) ---
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,                  # 3 epochs
    per_device_train_batch_size=2,       # Per-device batch size of 2
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # Gradient accumulation over four steps
    warmup_steps=100,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    learning_rate=2e-4,                  # Learning rate 2×10⁻⁴
    fp16=False,
    bf16=True,                           # bf16 precision
    group_by_length=True,
    report_to="none",
    gradient_checkpointing=True,         # Enable gradient checkpointing
    optim="adamw_torch",                 # AdamW optimizer
    disable_tqdm=True, 
)

# --- 6. Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training completed!")

# --- 7. Save model ---
print("💾 Saving model...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ LoRA adapters saved to: {output_dir}")